# 📗 Self-reflection 루프

지금까지 우리는 도구를 정의해 **에이전트에 연결**하고(에이전트·도구), 에이전트의 **사고 루프**를 해부했으며(ReAct), 앞 단원에서는 **계획을 세워** 수집부터 리포트까지 자동 분석을 시켰습니다. 그런데 한 가지 문제가 남습니다. **모델이 한 번에 내놓은 답이 늘 좋은 건 아니라는 것**입니다.

이번 시간엔 사람이 초안을 쓰고 스스로 고쳐 쓰듯, 모델이 **자기 출력을 스스로 비평하고 다시 쓰게** 만드는 **Self-reflection(자기 성찰) 루프**(**생성 → 비평 → 수정**)를 만듭니다. 그리고 이 루프를 오늘 제공하는 분석 도구와 붙여 **인사이트 리포트를 자동으로** 만들어 냅니다.

## ⏪ 복습

- **모델 호출**(LangChain 기본 단원): `ChatOpenAI(...)` 로 챗 모델을 만들고 `.invoke(메시지)` 로 답을 받았습니다(`.text` 로 본문). 오늘은 그 모델을 준비 셀 다음에서 `model` 로 만들어 두고 그대로 씁니다.
- **구조화된 출력**(에이전트·도구 단원): `with_structured_output(스키마)` 로 모델 답을 자유 문장이 아니라 **정해진 구조**(필드가 있는 객체)로 받았습니다. 오늘은 이 스킬을 **비평 스키마**로 심화 재사용합니다. 새로 배우는 문법이 아닙니다.
- **분석 도구**(오늘 제공): pandas 집계를 함수로 감싸 **요약 문자열**을 돌려주는 도구입니다. 4절 실습 파일에 들어 있습니다.
- **인사이트 리포트 골격**(데이터 분석 종합실습 단원): 숫자 재진술 → 해석 → 실행 제안. 관찰된 데이터일 뿐 **인과는 아님**.

**오늘의 목표**

- [ ] 한 번의 생성이 **왜 부족한지** 예시로 확인한다.
- [ ] 모델 출력을 **구조화된 비평**(점수·개선점)으로 평가한다.
- [ ] **생성 → 비평 → 수정** 을 **반복 루프**로 묶어(임계 점수·최대 반복) 품질을 끌어올린다.
- [ ] 반복별 **점수 이력**을 남겨 개선 과정을 관찰한다.
- [ ] 분석 도구와 결합해 **인사이트 리포트를 자동 생성**하고 **파일로 남긴다**.

아래 준비 셀을 먼저 실행하세요. **지금까지 쓰던 본인 OpenAI API 키가 필요합니다.** 일차 폴더에서 `.env.example` 을 `.env` 로 복사하고 `OPENAI_API_KEY` 를 채우면 됩니다. 오늘은 성찰 루프가 한 리포트에 모델을 여러 번 부르지만, 짧은 문장이라 비용은 여전히 아주 적습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우
load_dotenv("../../.env") # 교안 폴더 안의 정답 폴더에서 실행하는 경우
load_dotenv("../../../.env") # 교안 폴더 안의 과제/정답 에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델 - 실행만 하세요(LangChain 기본 단원에서 만든 것과 같습니다).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('모델 준비 완료:', type(model).__name__)

---
# 1. 한 번의 생성은 왜 부족할까

## 왜 필요할까요?
사람도 글을 **한 번에** 완성하지 않습니다. 초안을 쓰고, 읽어 보고, 부족한 곳을 고칩니다. 언어모델도 마찬가지입니다. 처음 답은 그럴듯해 보여도 **구체적 수치가 빠지거나, 해석이 뻔하거나, 실행 제안이 없는** 경우가 많습니다.

먼저 수치 요약 하나를 주고 **초안**을 받아 봅니다. 그다음 이 초안이 정말 좋은지 따져볼 겁니다.

<img src="../images/one_shot_vs_reflect.png" width="820">

*왼쪽이 지금 해 볼 방식입니다. 오른쪽처럼 만드는 것이 오늘의 목표고, 차이는 **비평과 수정을 한 번 더 거치느냐** 하나입니다.*

In [ ]:
# 리포트의 재료가 될 수치 요약 - 아직 해석도 제안도 없는 '숫자 그대로'다
summary = '카테고리별 평균 완료율: 파이썬 70.5%, 웹개발 69.8%, 데이터분석 61.7%, 디자인 56.9%, AI머신러닝 54.8%. 전체 평균 만족도 4.1/5.0.'
print(summary)

In [ ]:
# 생성 - 수치 요약을 받아 인사이트 리포트 초안을 쓴다
# 기준(세 문장·한국어)을 system 에 못 박아야 매번 같은 모양의 초안이 나온다
GEN_SYSTEM = '너는 데이터 분석 리포트 작성자다. 주어진 수치 요약을 바탕으로 핵심 인사이트를 한국어 세 문장으로 써라.'

def generate(summary):
    """수치 요약 문자열을 받아 리포트 초안(문자열)을 돌려준다."""
    r = model.invoke(
        [{'role': 'system', 'content': GEN_SYSTEM},
         {'role': 'user', 'content': summary}])
    return r.text        # 답 객체가 아니라 본문 문자열만 넘겨야 다음 단계가 그대로 받는다

In [ ]:
# 한 번만 부른 초안 - 이 글의 부족한 점이 오늘 루프를 만드는 이유가 된다
draft = generate(summary)
print(draft)

**초안을 직접 읽고 세 가지를 확인해 보세요.**

1. **구체적 수치**를 인용했나요, 아니면 "높은 편"·"어느 정도" 로 뭉갰나요?
2. 수치의 **의미를 해석**했나요, 아니면 숫자를 다시 읽기만 했나요?
3. **무엇을 하라는 제안**이 있나요?

셋 중 하나라도 비어 있다면 그게 바로 **한 번의 생성으로는 부족한 지점**입니다. 다만 이 판정을 사람이 매번 눈으로 하는 대신, 다음 절에서 **모델에게 같은 기준으로 채점시켜** 자동으로 고치게 만듭니다. 그러면 점수라는 **숫자**가 남아 개선 여부를 추적할 수 있습니다.

### ✅ 바로 확인 퀴즈

**1.** Self-reflection 루프가 필요한 이유를 한 문장으로 말해 보세요.

<details><summary>정답 보기</summary>

모델이 **한 번에 내놓은 답은 구체성·해석·실행 제안이 부족할 수 있어**, 스스로 비평하고 다시 쓰게 하면 품질이 올라가기 때문입니다.

</details>

**2.** 같은 `summary` 로 `generate` 를 두 번 부르면 글자까지 똑같은 초안이 나올까요?

<details><summary>정답 보기</summary>

아닙니다. 모델 출력은 부를 때마다 표현이 달라집니다. 그래서 초안을 **눈으로만** 판정하면 기준이 흔들리고, 다음 절처럼 **점수라는 숫자**로 받아야 좋아졌는지 비교할 수 있습니다.

</details>

---
# 2. 비평을 구조로 받기: 점수와 개선점

## 왜 구조화할까요?
비평을 자유 문장으로 받으면 사람이 읽기엔 좋지만 **프로그램이 판단하기는 어렵습니다**. "점수가 8점 이상이면 멈춰라" 같은 조건을 걸려면 **점수를 숫자로**, 개선점을 **목록으로** 꺼낼 수 있어야 합니다. 그래서 앞서 익힌 **구조화된 출력**(`with_structured_output`)을 그대로 다시 씁니다. 답의 모양을 미리 정해 두고 그 틀로 받는 것입니다. 문법은 이미 배웠으니, 여기서는 **비평용 스키마**로 응용하는 게 핵심입니다.

## 문법: 구조 정의와 구조화 출력
- `class Critique(BaseModel)` 로 받을 **필드**(`score`·`issues`)를 선언합니다.
- `model.with_structured_output(Critique)` 로 모델이 **그 구조로** 답하게 만듭니다.
- 결과는 `.score`(정수)·`.issues`(리스트)로 바로 꺼내 씁니다.

In [ ]:
# 비평 결과를 담을 구조 - 점수(1~10)와 개선점 목록
from pydantic import BaseModel, Field

class Critique(BaseModel):
    # ge/le 로 범위를 걸어 두면 모델이 0 이나 100 같은 값을 돌려주지 못한다
    score: int = Field(ge=1, le=10, description='1~10 종합 점수')
    # description 은 모델이 읽는 지시문이다 - 여기를 비우면 개선점이 장황해진다
    issues: list[str] = Field(description='개선점 목록(짧게)')

In [ ]:
# 비평 - 리포트를 구조화된 출력(Critique)으로 평가한다
# 채점 기준 세 가지를 여기 적어 둬야 매번 같은 잣대로 점수가 나온다
CRITIC_SYSTEM = '너는 깐깐한 리포트 편집자다. 아래 리포트를 평가하라. 구체적 수치 인용·해석의 명확성·실행 제안 유무를 기준으로 1~10점을 매기고, 개선점을 항목으로 지적하라.'

def critique(report):
    """리포트를 평가해 Critique(점수·개선점) 객체를 돌려준다."""
    # 원래 model 은 그대로 두고 '구조로 답하는 새 모델'을 받는다 - model 자체는 바뀌지 않는다
    critic_model = model.with_structured_output(Critique)
    return critic_model.invoke(
        [{'role': 'system', 'content': CRITIC_SYSTEM},
         {'role': 'user', 'content': report}])

In [ ]:
# 방금 그 초안을, 같은 세 기준(수치 인용·해석·제안)으로 모델에게 채점시킨다
critique_result = critique(draft)
print('점수:', critique_result.score)
print('개선점:')
for issue in critique_result.issues:
    print(' -', issue)

> 점수가 **숫자**라서 `if critique_result.score >= 8:` 같은 **판단**에 바로 쓸 수 있고, 개선점이 **리스트**라서 다음 단계(수정)에 그대로 넘길 수 있습니다. 이게 구조화의 힘입니다.

### 🖐️ 함께 따라하기: 다른 글을 같은 잣대로 재기

이번엔 데모와 **다른 글**에 같은 비평을 적용해 봅니다. 아래 `weak_draft` 는 수치도 제안도 없는 부실한 리포트입니다. 이걸 `critique` 로 평가해 **점수**를 출력하고, 8점 이상이면 `'충분'`, 아니면 `'수정 필요'` 를 함께 출력해 보세요.

**확인 기준**: 앞 데모의 초안 점수보다 **낮게** 나오면, 비평 함수가 글의 품질을 실제로 구분하고 있다는 뜻입니다.

In [ ]:
# 데모의 초안과 견줄 '부실한 글' - 수치도 해석도 제안도 없다
weak_draft = '수강생들이 강의를 듣는다. 완료율은 카테고리마다 다르다. 만족도도 괜찮은 편이다.'
print(weak_draft)

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) critique(weak_draft) 로 평가 결과를 받는다
# 2) 그 결과의 score 를 출력한다
# 3) 8 이상이면 '충분', 아니면 '수정 필요' 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 비평을 자유 문장이 아니라 **구조**(점수·개선점)로 받으면 무엇이 좋아지나요?

<details><summary>정답 보기</summary>

점수를 **숫자로 판단**(임계값 비교)하고 개선점을 **목록으로 다음 단계에 전달**할 수 있어, 루프의 종료 조건과 수정 입력으로 바로 쓸 수 있습니다.

</details>

**2.** 모델이 정해진 구조로 답하게 만드는 메서드는 무엇인가요?

<details><summary>정답 보기</summary>

**`with_structured_output(스키마)`** 입니다. 스키마(BaseModel)로 답의 모양을 정해 두고 그 틀로 받습니다.

</details>

---
# 3. 생성 → 비평 → 수정 루프

## 왜 루프일까요?
한 번 고친다고 완벽해지지 않습니다. **비평 → 수정 → 다시 비평**을 **점수가 충분해질 때까지**(또는 **정해진 횟수만큼**) 반복합니다. 사람의 퇴고와 똑같은 구조입니다.

## 두 가지 종료 조건 (둘 다 필요)
- **임계 점수**: 점수가 목표(예: 8점) 이상이면 **성공적으로 멈춘다**.
- **최대 반복**: 아무리 고쳐도 안 오를 수 있으니 **횟수 상한**(예: 3회)으로 **무한 루프를 막는다**.

먼저 **수정 함수**를 만들고, 세 함수를 하나의 루프로 묶습니다.

<img src="../images/reflection_loop.png" width="860">

*되돌아오는 화살표가 이 그림의 핵심입니다. **수정한 글을 다시 비평**하기 때문에 반복이 됩니다. 위로 빠져나가는 화살표(임계 점수 도달)와 최대 반복이 그 고리를 끊는 두 가지 방법입니다.*

In [ ]:
# 수정 - 리포트와 개선점을 받아 다시 쓴다
# 새로 쓰지 말고 '고쳐 쓰라'고 해야 앞 초안의 좋은 부분이 살아남는다
REVISE_SYSTEM = '너는 리포트 작성자다. 아래 [리포트]를 [개선점]을 모두 반영해 다시 써라. 한국어 세 문장을 유지하라.'

def revise(report, issues):
    """리포트와 개선점 목록을 받아 고쳐 쓴 리포트(문자열)를 돌려준다."""
    # 개선점은 줄바꿈 목록으로 펴서 넘긴다 - 리스트를 그대로 넣으면 모델이 파이썬 표기를 읽게 된다
    user = f'[리포트]\n{report}\n\n[개선점]\n' + '\n'.join(f'- {x}' for x in issues)
    r = model.invoke(
        [{'role': 'system', 'content': REVISE_SYSTEM},
         {'role': 'user', 'content': user}])
    return r.text

In [ ]:
# 생성 -> (비평 -> 수정) 반복. 반복 횟수에 상한이 있으니 while 이 아니라 for 로 쓴다.
def reflect(summary, threshold=8, max_iter=3):
    """(최종 리포트, 점수 이력) 을 돌려준다."""
    report = generate(summary)      # 생성은 맨 처음 한 번뿐이고 뒤로는 고쳐 쓰기만 한다
    score_history = []              # 반복마다 점수를 쌓아야 좋아졌는지 눈으로 확인된다
    for _ in range(max_iter):       # 상한이 없으면 점수가 안 오를 때 영영 끝나지 않는다
        result = critique(report)
        score_history.append(result.score)
        if result.score >= threshold:
            break                   # 충분하면 수정하지 않고 그대로 내보낸다
        report = revise(report, result.issues)
    return report, score_history

In [ ]:
# 이제 생성·비평·수정이 한 함수 안에서 자동으로 돈다 - 사람은 요약만 넣는다
final_report, score_history = reflect(summary)
print('점수 이력:', score_history)
print()
print('최종 리포트:')
print(final_report)

> 점수 이력이 예를 들어 `[4, 8]` 이면, **초안 4점 → 한 번 수정 후 8점**으로 올라 임계값에 도달해 멈춘 것입니다. 이 **점수 이력**이 루프가 실제로 품질을 끌어올렸는지 보여 주는 **증거**입니다.

### 🖐️ 함께 따라하기: 다른 요약에 임계값을 높여 보기

아래 `weekly_summary` 는 데모와 **다른 수치 요약**(주간 학습 지표)입니다. 이 요약으로 `reflect(weekly_summary, threshold=10, max_iter=3)`(**도달하기 어려운 임계값**입니다)을 호출해 점수 이력의 **길이**를 출력해 보세요(끝까지 못 도달하면 max_iter 만큼 반복합니다).

**확인 기준**: 길이가 대개 **3**(= `max_iter`)으로 나옵니다. 10 점은 웬만해선 나오지 않아 루프가 끝까지 돌기 때문입니다(운 좋게 10 점을 받으면 더 짧아질 수도 있습니다). **멈추는 조건이 둘(점수 도달 · 횟수 소진)이라는 것**, 그래서 무한히 고쳐 쓰지 않는다는 것이 이 연습의 요점입니다.

In [ ]:
# 데모와 다른 수치 요약 - 같은 루프가 어떤 재료에나 붙는지 확인한다
weekly_summary = ('요일별 평균 학습 시간: 토 62분, 일 58분, 수 41분, 월 39분. '
                  '주간 완주 인원 128명, 지난주 대비 12명 증가.')
print(weekly_summary)

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) reflect(weekly_summary, threshold=10, max_iter=3) 을 호출한다
# 2) 반환된 점수 이력의 길이를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 루프에 **최대 반복 횟수**를 두는 이유는?

<details><summary>정답 보기</summary>

점수가 임계값에 **영영 도달하지 못할 수도** 있어, 상한이 없으면 루프가 끝나지 않습니다. 최대 반복으로 **무한 루프를 막습니다**.

</details>

**2.** 점수 이력(`score_history`)은 무엇을 확인하는 데 쓰나요?

<details><summary>정답 보기</summary>

반복하면서 **품질(점수)이 실제로 올라갔는지** 관찰하는 데 씁니다. 개선 과정을 남기는 로그입니다.

</details>

---
# 4. 분석 도구 + 성찰 = 인사이트 리포트 자동 생성

지금까지는 요약 문자열을 사람이 넣고 생성·비평·수정을 차례로 불렀습니다.
이번에는 **앞 절에서 만든 성찰 루프까지 도구로 감싸** 데이터 요약·파일 저장 도구와 함께 에이전트 한 대에 붙입니다. 남이 준 도구만 쓰는 것이 아니라, **내가 만든 것도 도구가 됩니다.**
그러면 요청 문장 한 줄로 데이터에서 파일까지 이어지고, 어느 도구를 언제 부를지는 모델이 정합니다.

> **이 절은 `.py` 로 실습합니다.** 같은 폴더의 **`01_자동_리포트.py`** 를 열고 `uv run 01_자동_리포트.py` 로 돌리세요.

---
## 이번 강의 정리

| 단계 | 하는 일 | 핵심 |
|---|---|---|
| 생성 | 수치 요약으로 초안 작성 | `generate(summary)` |
| 비평 | 구조화된 점수·개선점 | `with_structured_output(Critique)` |
| 수정 | 개선점을 반영해 다시 쓰기 | `revise(report, issues)` |
| 루프 | 임계 점수·최대 반복까지 반복 | `reflect(...)` → `(리포트, 점수이력)` |
| 결합 | 요약·성찰·저장을 도구로 만들어 에이전트 한 대에 | 4절 실습 `01_자동_리포트.py` |

- Self-reflection = **생성 → 비평 → 수정**의 반복. 종료 조건은 **임계 점수**와 **최대 반복** 둘 다.
- 비평을 **구조화**해야 점수로 판단하고 개선점을 다음 단계로 넘길 수 있습니다.
- 요약·성찰·저장을 **모두 도구로** 만들어 붙이면, 요청 문장 한 줄로 **데이터에서 파일까지** 이어집니다.
- 다만 비평은 **수치를 인용했는지**만 볼 뿐 **그 수치가 맞는지**는 확인하지 못합니다. 점수가 올라도 리포트의 숫자는 사람이 원본 요약과 대조해야 합니다.

## ⏭️ 예고

루프가 여러 번 모델을 호출하는 만큼, **비용·실패·느려짐**이 눈에 안 보이면 운영이 어렵습니다. 다음 시간엔 에이전트 안에서 무슨 일이 일어나는지 **들여다보는 관측성(Langfuse)** 을 배웁니다. 실행을 대시보드에 남기고, 토큰과 비용을 직접 계산해 서버 값과 맞춰 보고, 프롬프트를 버전으로 관리하고, 기록을 읽어 틀어진 답을 고칩니다.

수고하셨습니다!